In [1]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as ticker
import seaborn as sns
import os
import re
import json
import unicodedata
from sqlalchemy import create_engine, text, inspect
from IPython.display import display

palette_rgb = [(61, 183, 228), (255, 136, 73), (105, 190, 40)]
palette = [(r/255, g/255, b/255) for r, g, b in palette_rgb]

# 1. COMANDOS PARA RECARGA AUTOMÁTICA
%load_ext autoreload
%autoreload 2
%matplotlib inline

pd.set_option('display.float_format', lambda x: f'{x:.4f}') # evitar notación cientifica y mostrar 2 decimales
pd.set_option('display.max_columns', None)   # mostrar todas las columnas
pd.set_option('display.width', 0)           # dejar que use todo el ancho disponible
pd.set_option('display.max_colwidth', None) # Quitar el límite de ancho de las columnas
pd.set_option('display.expand_frame_repr', False) # Para que no "envuelva" la tabla y se mantenga en una sola fila larga


# Descarga de bases de datos:

In [2]:
# Conexión persistente
DB_URL = "sqlite:///../.data/flight_account_001_xauusd.db"
engine = create_engine(DB_URL, connect_args={'timeout': 15})

def extract_trading_ecosystem(db_engine) -> dict:
    """
    Extrae la topología completa en una sola transacción I/O.
    Retorna diccionario con las 4 tablas crudas y la Matriz Analítica Maestra.
    """
    # Verificación de existencia de tabla mediante API pública
    insp = inspect(db_engine)
    has_layer = insp.has_table("analysis_layer")
    
    queries = {
        "unified": "SELECT * FROM unified_department WHERE asset LIKE '%XAUUSD%';",
        "layer": "SELECT * FROM analysis_layer;" if has_layer else "SELECT 1;",
        "efficiency": "SELECT * FROM efficiency_audit;",
        "tactical": "SELECT * FROM tactical_audit;",
        "master": """
            SELECT 
                u.id, u.asset, u.market_bias, u.calc_edge, u.long_prob, u.short_prob, u.created_at as edge_time,
                e.bias_a, e.real_bias_b, e.resolution_type, e.false_regime_rate,
                t.tier_setup, t.market_state, t.entry_time, t.exit_time, 
                t.r_r, t.pnl_and_cost, t.mfe_favorable, t.mae_adverse, t.compliance
            FROM unified_department u
            LEFT JOIN efficiency_audit e ON u.id = e.id
            LEFT JOIN tactical_audit t ON u.id = t.id
            WHERE u.asset LIKE '%XAUUSD%'
        """
    }
    
    db_data = {}
    with db_engine.connect() as conn:
        for name, query in queries.items():
            try:
                if name == "master":
                    df = pd.read_sql(text(query), conn, parse_dates=['edge_time', 'entry_time', 'exit_time'])
                else:
                    df = pd.read_sql(text(query), conn)
                db_data[name] = df
            except Exception as e:
                print(f"Advertencia: Fallo al extraer {name}. Detalle: {e}")
                db_data[name] = pd.DataFrame()

    df_master = db_data["master"]

    # 1. Validación Estricta de Tipos Numéricos en Matriz (Fail-Fast)
    numeric_cols = ['calc_edge', 'long_prob', 'short_prob', 'r_r', 'pnl_and_cost', 'mfe_favorable', 'mae_adverse']
    for col in numeric_cols:
        if col in df_master.columns:
            try:
                df_master[col] = pd.to_numeric(df_master[col], errors='raise')
            except ValueError as e:
                raise TypeError(f"Corrupción de datos en {col}: {e}. Purgar SQLite.")

    # 2. Control de Causalidad (Fuga de Datos)
    time_leaks = df_master[df_master['entry_time'].notna() & (df_master['entry_time'] < df_master['edge_time'])]
    if not time_leaks.empty:
        raise ValueError(f"Fuga Lógica Crítica: {len(time_leaks)} ejecuciones previas al Edge.")

    return db_data

def compute_structural_kpis(df: pd.DataFrame) -> pd.DataFrame:
    """Evalúa la Esperanza Matemática pura del sistema táctico."""
    exec_df = df[df['compliance'].isin(['Edge_valid', 'Invalid_edge']) & df['r_r'].notna()].copy()
    
    if exec_df.empty:
        return pd.DataFrame([{"Error": "Sin ejecuciones completadas para medir KPIs"}])

    wins = exec_df[exec_df['r_r'] > 0]
    losses = exec_df[exec_df['r_r'] <= 0]
    
    win_sum = wins['pnl_and_cost'].sum()
    loss_sum = abs(losses['pnl_and_cost'].sum())
    
    metrics = {
        'N_Trades': len(exec_df),
        'Win_Rate_%': round((len(wins) / len(exec_df)) * 100, 2),
        'Expectancy_R': round(exec_df['r_r'].mean(), 2),
        'Profit_Factor': round(win_sum / loss_sum, 2) if loss_sum != 0 else float('inf'),
        'MFE_Promedio': round(exec_df['mfe_favorable'].mean(), 2),
        'MAE_Promedio': round(exec_df['mae_adverse'].mean(), 2)
    }
    return pd.DataFrame([metrics])

def evaluate_predictive_edge(df: pd.DataFrame) -> pd.DataFrame:
    """Mide la correlación predictiva entre la hipótesis matemática y la resolución de mercado."""
    eval_df = df[df['bias_a'].notna() & df['real_bias_b'].notna()].copy()
    
    if eval_df.empty:
        return pd.DataFrame([{"Error": "Muestra insuficiente en Efficiency Audit para correlación."}])
    
    # Target Binario: 1 si el sesgo A es igual a la realidad B, 0 si falló.
    eval_df['edge_accurate'] = np.where(eval_df['bias_a'] == eval_df['real_bias_b'], 1, 0)
    
    # Correlación de Pearson
    matrix_cols = ['calc_edge', 'long_prob', 'short_prob', 'edge_accurate']
    corr_matrix = eval_df[matrix_cols].corr()
    
    target_corr = corr_matrix[['edge_accurate']].drop('edge_accurate').rename(columns={'edge_accurate': 'Pearson_Correlation_Target'})
    return target_corr

def analyze_trading_system(df):
    # ---------------------------------------------------------
    # PRE-PROCESAMIENTO DE DATOS
    # ---------------------------------------------------------
    df['pnl_and_cost'] = pd.to_numeric(df['pnl_and_cost'], errors='coerce')
    
    # Derivar dirección teórica y probabilidad dominante
    df['theoretical_dir'] = np.where(df['calc_edge'] >= 0, 'Long', 'Short')
    df['primary_prob'] = np.where(df['theoretical_dir'] == 'Long', df['long_prob'], df['short_prob'])
    df['calc_edge_mag'] = df['calc_edge'].abs()
    
    # ---------------------------------------------------------
    # BLOQUE 1: KPIs DEL EDGE ESTRUCTURAL (Modelo Predictivo)
    # ---------------------------------------------------------
    # Filtrar solo resoluciones estructurales definidas
    df_struct = df[df['specific_bias_compliance'].isin(['Valid', 'Invalid'])].copy()
    df_struct['is_structurally_valid'] = np.where(df_struct['specific_bias_compliance'] == 'Valid', 1, 0)
    
    struct_total = len(df_struct)
    struct_win_rate = df_struct['is_structurally_valid'].mean()
    
    print("=== BLOQUE 1: EDGE ESTRUCTURAL (TEÓRICO) ===")
    print(f"Total Setups con Resolución Gráfica: {struct_total}")
    print(f"Win Rate Estructural (Acierto Geométrico): {struct_win_rate:.2%}")
    
    # Cuantiles para encontrar rangos óptimos
    print("\n--- Tasa de Acierto por Rangos de Probabilidad Dominante ---")
    bins_prob = pd.qcut(df_struct['primary_prob'], q=3, duplicates='drop')
    print(df_struct.groupby(bins_prob, observed=False)['is_structurally_valid'].agg(
        Total_Setups='count', 
        Win_Rate='mean'
    ).apply(lambda x: round(x, 4)))

    print("\n--- Tasa de Acierto por Magnitud de Calc_Edge ---")
    bins_edge = pd.qcut(df_struct['calc_edge_mag'], q=3, duplicates='drop')
    print(df_struct.groupby(bins_edge, observed=False)['is_structurally_valid'].agg(
        Total_Setups='count', 
        Win_Rate='mean'
    ).apply(lambda x: round(x, 4)))

    # ---------------------------------------------------------
    # BLOQUE 2: KPIs DEL EDGE DE EJECUCIÓN (PnL Real)
    # ---------------------------------------------------------
    df_exec = df.dropna(subset=['pnl_and_cost']).copy()
    df_exec['is_win'] = np.where(df_exec['pnl_and_cost'] > 0, 1, 0)
    
    exec_total = len(df_exec)
    exec_win_rate = df_exec['is_win'].mean()
    avg_win = df_exec[df_exec['is_win'] == 1]['pnl_and_cost'].mean()
    avg_loss = df_exec[df_exec['is_win'] == 0]['pnl_and_cost'].mean()
    expectancy = (exec_win_rate * avg_win) + ((1 - exec_win_rate) * avg_loss)
    
    print("\n=== BLOQUE 2: EDGE DE EJECUCIÓN (PRÁCTICO) ===")
    print(f"Total Ejecuciones Loggeadas: {exec_total}")
    print(f"Win Rate Operativo (PnL > 0): {exec_win_rate:.2%}")
    print(f"Promedio Ganancia: {avg_win:.2f}")
    print(f"Promedio Pérdida: {avg_loss:.2f}")
    print(f"Esperanza Matemática PnL (EV): {expectancy:.2f}")
    
    # ---------------------------------------------------------
    # BLOQUE 3: MATRIZ DE CORRELACIÓN EN CONTEXTO VÁLIDO
    # ---------------------------------------------------------
    df_valid = df[df['compliance'] == 'Edge_valid'].copy()
    corr_valid = df_valid[['primary_prob', 'calc_edge_mag', 'pnl_and_cost']].corr()
    
    print("\n=== BLOQUE 3: CORRELACIÓN (SOLO COMPLIANCE VÁLIDO) ===")
    print(corr_valid)

# Analisis de datos - MIO 

In [3]:
db_data = extract_trading_ecosystem(engine)

unified_df = db_data["unified"]
layer_df = db_data["layer"]
efficiency_df = db_data["efficiency"]
tactical_df = db_data["tactical"]
df_master = db_data["master"]

In [4]:
# The columns you want to spread out
cols = ["direction", "strength", "score", "thesis"]

# 1. You MUST pivot on 'trade_id' to group the layers together
wide = (
    layer_df.pivot(
        index="trade_id", 
        columns="layer_name",
        values=cols
    )
)

# 2. Flatten the MultiIndex columns
wide.columns = [
    f"{layer.lower()}_{col}"
    for col, layer in wide.columns
]

# 3. Bring trade_id back as a regular column
wide = wide.reset_index()

# 4. RENAME 'trade_id' to 'id' so your final output matches your desired naming
wide = wide.rename(columns={"trade_id": "id"})

# 5. Reorder the columns
wide = wide[['id', 'p0_direction', 'p1_direction', 'p2_direction', 'p3_direction', 'p4_direction',
            'p0_strength', 'p1_strength', 'p2_strength', 'p3_strength', 'p4_strength',
            'p0_score', 'p1_score', 'p2_score', 'p3_score', 'p4_score']]

In [5]:
filt_unified = unified_df[['id','calc_edge', 'market_bias', 'long_prob', 'short_prob', 'no_trade_prob', 'edge_validation_price', 'structural_invalidation', 'trade_status', 'edge_description']]
filt_eff = efficiency_df[['id','resolution_type', 'false_regime_rate', 'specific_bias_compliance','created_at', 'bias_a', 'real_bias_b', 'efficiency_timeframe']].sort_values(by='created_at')
filt_tact = tactical_df[['id','compliance', 'entry_time', 'exit_time', 'trade_decision', 'pnl_and_cost','size', 'mae_adverse', 'mfe_favorable', 'session']]

# 1. Merge ALL the dataframes (Notice we added `.merge(wide...)` at the end of this line!)
merged = filt_unified.merge(filt_eff, on='id', how='left') \
                     .merge(filt_tact, on='id', how='left') \
                     .merge(wide, on='id', how='left')

# 2. Select your columns - Now pandas will find the P0-P4 columns because 'wide' is merged in
columns_to_keep = [
    'id', 'long_prob', 'short_prob', 'no_trade_prob', 'calc_edge', 'market_bias', 'efficiency_timeframe', 'edge_validation_price',
    'structural_invalidation', 'bias_a', 'real_bias_b', 'resolution_type', 'specific_bias_compliance', 'false_regime_rate', 'compliance', 'trade_status', 'size', 'pnl_and_cost', 
    'created_at', 'entry_time', 'exit_time', 'trade_decision', 'edge_description',
    'p0_direction', 'p1_direction', 'p2_direction', 'p3_direction', 'p4_direction',
    'p0_strength', 'p1_strength', 'p2_strength', 'p3_strength', 'p4_strength',
    'p0_score', 'p1_score', 'p2_score', 'p3_score', 'p4_score'
]

# Apply the column filter
merged = merged[columns_to_keep]

# 3. Filtrado de los analisis que ya estén cerrados
merged = merged[merged['resolution_type'] != 'Open']

In [7]:
analyze_trading_system(merged)

=== BLOQUE 1: EDGE ESTRUCTURAL (TEÓRICO) ===
Total Setups con Resolución Gráfica: 36
Win Rate Estructural (Acierto Geométrico): 66.67%

--- Tasa de Acierto por Rangos de Probabilidad Dominante ---
                Total_Setups  Win_Rate
primary_prob                          
(0.099, 0.156]            15    0.5333
(0.156, 0.237]             9    0.6667
(0.237, 0.408]            12    0.8333

--- Tasa de Acierto por Magnitud de Calc_Edge ---
                Total_Setups  Win_Rate
calc_edge_mag                         
(-0.001, 0.2]             15    0.5333
(0.2, 0.433]               9    0.6667
(0.433, 0.825]            12    0.8333

=== BLOQUE 2: EDGE DE EJECUCIÓN (PRÁCTICO) ===
Total Ejecuciones Loggeadas: 19
Win Rate Operativo (PnL > 0): 36.84%
Promedio Ganancia: 35.23
Promedio Pérdida: -17.32
Esperanza Matemática PnL (EV): 2.04

=== BLOQUE 3: CORRELACIÓN (SOLO COMPLIANCE VÁLIDO) ===
               primary_prob  calc_edge_mag  pnl_and_cost
primary_prob         1.0000         0.9952    

# Analisis de Datos - GEMINI

In [8]:
db_data = extract_trading_ecosystem(engine)

unified_df = db_data["unified"]
layer_df = db_data["layer"]
efficiency_df = db_data["efficiency"]
tactical_df = db_data["tactical"]
df_master = db_data["master"]

print(f"Dimensiones extraídas | Master: {df_master.shape} | Unified: {unified_df.shape} | Tactical: {tactical_df.shape}")

print("\n--- [CAPA 3] KPIs Tácticos de Ejecución ---")
kpis = compute_structural_kpis(df_master)
display(kpis)

print("\n--- [CAPA 3] Correlación Predictiva del Algoritmo (calc_edge) ---")
edge_corr = evaluate_predictive_edge(df_master)
display(edge_corr)

Dimensiones extraídas | Master: (37, 20) | Unified: (37, 23) | Tactical: (36, 60)

--- [CAPA 3] KPIs Tácticos de Ejecución ---


,N_Trades,Win_Rate_%,Expectancy_R,Profit_Factor,MFE_Promedio,MAE_Promedio
0,18,100.0000,2.5300,inf,1.5100,0.7000



--- [CAPA 3] Correlación Predictiva del Algoritmo (calc_edge) ---


,Pearson_Correlation_Target
calc_edge,-0.1032
long_prob,-0.0304
short_prob,0.1677


In [9]:
def analyze_structural_clusters(df: pd.DataFrame):
    """
    Agrupa y evalúa la eficiencia táctica por tipo de Setup y Estado de Mercado.
    Aísla las variables con verdadero impacto en el Expectancy.
    """
    # Filtrar únicamente operaciones ejecutadas
    exec_df = df[df['r_r'].notna()].copy()
    
    if exec_df.empty:
        return "Muestra táctica vacía."

    # Agrupación por Tier de Setup
    tier_group = exec_df.groupby('tier_setup').agg(
        N_Trades=('id', 'count'),
        Expectancy_R=('r_r', 'mean'),
        Max_MFE=('mfe_favorable', 'max'),
        Avg_MAE=('mae_adverse', 'mean')
    ).round(2).sort_values('Expectancy_R', ascending=False)
    
    # Agrupación por Estado de Mercado
    state_group = exec_df.groupby('market_state').agg(
        N_Trades=('id', 'count'),
        Expectancy_R=('r_r', 'mean'),
        Avg_MAE=('mae_adverse', 'mean')
    ).round(2).sort_values('Expectancy_R', ascending=False)

    return tier_group, state_group

# EJECUCIÓN 
tier_metrics, state_metrics = analyze_structural_clusters(df_master)

print("--- [CAPA 4] Eficiencia por Setup (Tier) ---")
display(tier_metrics)

print("\n--- [CAPA 4] Eficiencia por Estado de Mercado ---")
display(state_metrics)

--- [CAPA 4] Eficiencia por Setup (Tier) ---


,N_Trades,Expectancy_R,Max_MFE,Avg_MAE
tier_setup,,,,
F,2,3.4000,0.4200,1.0000
D,5,2.4700,2.7300,0.9000
B,4,2.4300,2.8300,0.2000
A,3,2.3900,4.5700,0.2800
C,5,2.3300,6.2600,1.0100



--- [CAPA 4] Eficiencia por Estado de Mercado ---


,N_Trades,Expectancy_R,Avg_MAE
market_state,,,
Range,10,2.6100,0.5800
Trend,9,2.4000,0.8200


In [10]:
def isolate_corrupt_records(df: pd.DataFrame):
    """
    Escanea la Matriz Analítica en busca de imposibilidades matemáticas.
    Condición de Fallo: El retorno (r_r) es mayor que la excursión máxima favorable (mfe_favorable).
    """
    # Filtrar registros con datos tácticos
    eval_df = df[df['r_r'].notna() & df['mfe_favorable'].notna()].copy()
    
    # Identificar corrupción lógica
    corrupt_mask = eval_df['r_r'] > eval_df['mfe_favorable']
    corrupt_trades = eval_df[corrupt_mask]
    
    # Extraer variables clave para depuración manual
    debug_cols = ['id', 'tier_setup', 'market_state', 'r_r', 'mfe_favorable', 'mae_adverse']
    
    return corrupt_trades[debug_cols]

# EJECUCIÓN
bad_records = isolate_corrupt_records(df_master)

print(f"--- [ALERTA] Se detectaron {len(bad_records)} registros matemáticamente imposibles ---")
display(bad_records)

--- [ALERTA] Se detectaron 12 registros matemáticamente imposibles ---


,id,tier_setup,market_state,r_r,mfe_favorable,mae_adverse
11,9fb17739-cb19-4771-836a-d3b6b28591bb,C,Trend,2.3380,0.5500,1.0000
15,7b6e9c10-50a6-4503-9044-04223b7a658b,C,Range,2.3991,0.4000,1.0000
16,42a48654-51d8-467a-81e1-1e5a230380df,F,Trend,2.7667,0.4200,1.0000
17,695b2b4b-fbc0-461e-a108-01576bad312e,D,Trend,2.2269,0.2500,1.0000
19,4b17b903-407d-4a5e-b238-1491ab64679b,D,Trend,2.2803,0.0000,0.9400
20,9602d430-4569-4682-b22c-9fb689d6fd4d,F,Range,4.0269,0.2000,1.0000
25,00e2e31b-5f14-4e36-959c-8c60b14d4e09,C,Trend,1.9930,0.1500,1.1500
27,ee712b3a-45ff-4fd6-a63e-a4c51ead6d15,D,Range,3.7330,0.2000,1.0000
31,b993a3b0-6309-42cd-a6fd-18fe953e3962,D,Trend,2.0474,0.2000,1.0000
32,d3c184ad-77bd-4005-a9af-d4d45a0056aa,A,Range,2.0853,0.0000,0.0000


In [11]:
import os
import sqlite3
import pandas as pd

db_path = "/home/jorgecg/projects/trading/blast_master/.data/flight_account_001_xauusd.db"
conn = sqlite3.connect(db_path)

# 1. ¿Qué tablas existen realmente en el archivo?
print("--- TABLAS EN LA BASE DE DATOS ---")
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
display(tables)

# 2. Esquema completo construido por SQLAlchemy
print("\n--- ESQUEMA DE TACTICAL_AUDIT ---")
if 'tactical_audit' in tables['name'].values:
    schema = pd.read_sql_query("PRAGMA table_info(tactical_audit)", conn)
    display(schema[['cid', 'name', 'type']])

# 3. Forzar el error para exponer la falla DDL
print("\n--- ERROR REAL DE INSERCIÓN ---")
try:
    conn.execute("""
        INSERT INTO tactical_audit (
            tactical_id, tier_setup, confirmation_status,
            g1_trend_15m, g2_fractal_trend, g3_limit_order, g4_breathing, g5_manual_cooldown, g6_sl_validated, g7_tp_validated
        ) VALUES (
            'test-uuid-1234', 'A', 'S7: No - forzó entrada pese a gate fallido (revenge)',
            0, 1, 1, 1, 1, 1, 1
        )
    """)
except Exception as e:
    print(f"Error nativo capturado: {type(e).__name__} -> {e}")
    
conn.close()

# 4. Auditoría de archivos en el directorio
print("\n--- ARCHIVOS EN EL DIRECTORIO .DATA ---")
data_dir = "/home/jorgecg/projects/trading/blast_master/.data/"
if os.path.exists(data_dir):
    for f in os.listdir(data_dir):
        print(f)

--- TABLAS EN LA BASE DE DATOS ---


,name
0,asset_balance
1,emotion_catalog
2,asset_config
3,unified_department
4,analysis_layer
5,efficiency_audit
6,tactical_audit



--- ESQUEMA DE TACTICAL_AUDIT ---


,cid,name,type
0,0,id,VARCHAR
1,1,compliance,VARCHAR
2,2,entry_time,DATETIME
3,3,exit_time,DATETIME
4,4,tier_setup,VARCHAR
5,5,market_state,VARCHAR
6,6,session,VARCHAR
7,7,exit_type,VARCHAR
8,8,trade_decision,VARCHAR
9,9,followed_plan,VARCHAR



--- ERROR REAL DE INSERCIÓN ---
Error nativo capturado: OperationalError -> table tactical_audit has no column named tactical_id

--- ARCHIVOS EN EL DIRECTORIO .DATA ---
flight_sessions.json
flight_account_001_xauusd.db-wal
flight_account_001_xauusd.db-shm
test_sessions.json
blast_master.db
flight_account_002_btcusdtp.db
flight_account_001_xauusd.db
archives
journal.db
backup.json
paused_audits.json
